# Solar Energy Forecaster - Step 1: Get the Data

**Goal:** Download weather + solar data for Kerala, clean it, and save it.

**What we're predicting:** How much sunlight hits a solar panel (measured in W/m²)

**What we use to predict:** 5 simple weather features:
1. Cloud cover (%) — clouds block sunlight
2. Temperature (°C) — hot clear days = more sun
3. Humidity (%) — humid air scatters light
4. Rain (mm) — rain means no sun
5. Hour of day (0-23) — sun is strongest at noon

---
## 1.1 Create project folders

In [ ]:
import os

# Create folders to organize our files
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

print("Folders created!")

---
## 1.2 Download data from Open-Meteo

Open-Meteo is a free weather API. No signup, no API key.

We ask it for one year of hourly data for Kochi, Kerala.

In [ ]:
import pandas as pd
import requests

# Where is our solar panel? (Kochi, Kerala)
LATITUDE = 9.93
LONGITUDE = 76.27

# What time period do we want?
START_DATE = "2022-01-01"
END_DATE = "2022-12-31"

# Ask Open-Meteo for the data
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": ",".join([
        "shortwave_radiation",    # Sunlight energy (this is what we PREDICT)
        "temperature_2m",         # Temperature
        "relative_humidity_2m",   # Humidity
        "cloud_cover",            # Cloud cover
        "precipitation",          # Rainfall
    ]),
    "timezone": "Asia/Kolkata"
}

print("Downloading data... (takes about 10 seconds)")
response = requests.get(url, params=params, timeout=60)

# Check if it worked
if response.status_code == 200:
    print("Download successful!")
else:
    print(f"Something went wrong. Error code: {response.status_code}")
    print(response.text[:300])

---
## 1.3 Convert the response to a table (DataFrame)

The API gives us JSON data. We convert it to a pandas DataFrame — basically a spreadsheet in Python.

In [ ]:
# The API response is JSON. Let's pull out the hourly data.
data = response.json()
hourly_data = data["hourly"]

# Convert to a DataFrame (like an Excel table)
df = pd.DataFrame(hourly_data)

# Look at what we got
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
df.head(10)

The `time` column is text like "2022-01-01T00:00". Let's convert it to a proper date/time.

In [ ]:
# Convert 'time' column from text to a real datetime
df["datetime"] = pd.to_datetime(df["time"])

# Make datetime the index (row label) of our table
df = df.set_index("datetime")

# Drop the old 'time' column (we don't need it anymore)
df = df.drop(columns=["time"])

# See the result
print(f"Date range: {df.index[0]} to {df.index[-1]}")
print(f"Total hours: {len(df)}")
print()
df.head()

---
## 1.4 Rename columns to simple names

The API column names are long. Let's make them shorter and easier to work with.

In [ ]:
df = df.rename(columns={
    "shortwave_radiation": "sunlight",       # Our TARGET (what we predict)
    "temperature_2m": "temperature",          # Feature 1
    "relative_humidity_2m": "humidity",        # Feature 2
    "cloud_cover": "clouds",                  # Feature 3
    "precipitation": "rain",                   # Feature 4
})

print("Columns after renaming:")
print(list(df.columns))
print()
df.head()

---
## 1.5 Save the raw data

Always save your raw data before making changes. If something goes wrong later, you can reload from here.

In [ ]:
df.to_csv("../data/raw/kerala_weather_2022.csv")
print("Raw data saved to: data/raw/kerala_weather_2022.csv")

---
## 1.6 Check for missing values

Real-world data often has gaps. Let's check if any hours are missing.

In [ ]:
# Count missing values in each column
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing: {missing.sum()}")

if missing.sum() == 0:
    print("\nGreat! No missing data.")
else:
    print("\nWe have some gaps. Let's fill them.")

In [ ]:
# Fill small gaps by estimating from neighboring values
# For example, if 2pm is missing but 1pm=500 and 3pm=600,
# we estimate 2pm as 550 (the middle value)
df = df.interpolate(method="linear", limit=3)

# If any gaps remain, use the value from the previous hour
df = df.ffill(limit=2)

# Drop any rows that STILL have missing values
rows_before = len(df)
df = df.dropna()
rows_after = len(df)

print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Dropped:     {rows_before - rows_after}")
print(f"Missing values now: {df.isnull().sum().sum()}")

---
## 1.7 Add the 'hour' feature

The hour of day is our 5th feature. We extract it from the datetime index.

Why is hour important? The sun follows a predictable daily pattern:
- Hours 0-5: nighttime, sunlight = 0
- Hours 6-8: sunrise, sunlight increasing
- Hours 10-14: peak sunlight
- Hours 16-18: sunset, sunlight decreasing
- Hours 19-23: nighttime, sunlight = 0

In [ ]:
# Extract hour from the datetime index
df["hour"] = df.index.hour

print("Columns now:")
print(list(df.columns))
print()
print("Sample data at different hours:")
sample = df["2022-03-15"]
print(sample[["hour", "sunlight", "clouds", "temperature"]].to_string())

---
## 1.8 Quick look at the data (statistics)

Before plotting, let's see the basic numbers.

In [ ]:
print("=== Summary Statistics ===")
print()
print(df.describe().round(1))
print()
print("--- What this tells us ---")
print(f"Sunlight ranges from {df['sunlight'].min():.0f} to {df['sunlight'].max():.0f} W/m²")
print(f"Temperature ranges from {df['temperature'].min():.1f} to {df['temperature'].max():.1f} °C")
print(f"Cloud cover ranges from {df['clouds'].min():.0f}% to {df['clouds'].max():.0f}%")
print(f"Humidity ranges from {df['humidity'].min():.0f}% to {df['humidity'].max():.0f}%")
print(f"\nAbout {(df['sunlight'] == 0).mean() * 100:.0f}% of hours have zero sunlight (nighttime)")

---
## 1.9 Visualize the data

Three plots to understand our data:
1. What does a full year look like?
2. How does a sunny week compare to a rainy week?
3. Which features are most related to sunlight?

In [ ]:
import matplotlib.pyplot as plt

# --- PLOT 1: Full year overview ---
# Resample to daily averages (easier to see patterns)
daily = df.resample("D").mean()

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Sunlight
axes[0].fill_between(daily.index, daily["sunlight"], color="orange", alpha=0.5)
axes[0].set_ylabel("Sunlight (W/m²)")
axes[0].set_title("Daily Average Sunlight - Kochi, Kerala (2022)", fontsize=14)
# Mark monsoon season
axes[0].axvspan("2022-06-01", "2022-09-30", alpha=0.1, color="blue", label="Monsoon season")
axes[0].legend()

# Clouds
axes[1].fill_between(daily.index, daily["clouds"], color="gray", alpha=0.5)
axes[1].set_ylabel("Cloud Cover (%)")
axes[1].set_title("Daily Average Cloud Cover")

# Rain
daily_rain = df["rain"].resample("D").sum()
axes[2].bar(daily_rain.index, daily_rain.values, width=1, color="steelblue", alpha=0.7)
axes[2].set_ylabel("Rain (mm/day)")
axes[2].set_title("Daily Total Rainfall")

# Temperature
axes[3].plot(daily.index, daily["temperature"], color="red", linewidth=1)
axes[3].set_ylabel("Temperature (°C)")
axes[3].set_title("Daily Average Temperature")

plt.tight_layout()
plt.savefig("../data/processed/yearly_overview.png", dpi=150)
plt.show()

print("\nNotice how sunlight DROPS during June-September (monsoon).")
print("That's when cloud cover and rainfall SPIKE.")
print("This pattern is exactly what our model will learn!")

In [ ]:
# --- PLOT 2: Sunny week vs Monsoon week ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Pick a clear week in March
sunny_week = df["2022-03-14":"2022-03-20"]
axes[0].plot(sunny_week.index, sunny_week["sunlight"], color="orange", linewidth=1.5)
axes[0].fill_between(sunny_week.index, sunny_week["sunlight"], alpha=0.2, color="orange")
axes[0].set_title("Sunny Week (March 14-20): Clear daily peaks", fontsize=12)
axes[0].set_ylabel("Sunlight (W/m²)")
axes[0].set_ylim(0, 1200)

# Pick a monsoon week in July
rainy_week = df["2022-07-11":"2022-07-17"]
axes[1].plot(rainy_week.index, rainy_week["sunlight"], color="orange", linewidth=1.5)
axes[1].fill_between(rainy_week.index, rainy_week["sunlight"], alpha=0.2, color="orange")
axes[1].set_title("Monsoon Week (July 11-17): Low and irregular sunlight", fontsize=12)
axes[1].set_ylabel("Sunlight (W/m²)")
axes[1].set_ylim(0, 1200)

plt.tight_layout()
plt.savefig("../data/processed/sunny_vs_monsoon.png", dpi=150)
plt.show()

print("Big difference! The model needs to learn BOTH patterns.")

In [ ]:
# --- PLOT 3: How does each feature relate to sunlight? ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Only look at daytime hours (when sunlight > 0)
daytime = df[df["sunlight"] > 0]

# Clouds vs Sunlight
axes[0, 0].scatter(daytime["clouds"], daytime["sunlight"], alpha=0.05, s=2, color="gray")
axes[0, 0].set_xlabel("Cloud Cover (%)")
axes[0, 0].set_ylabel("Sunlight (W/m²)")
axes[0, 0].set_title("More clouds = Less sunlight")

# Temperature vs Sunlight
axes[0, 1].scatter(daytime["temperature"], daytime["sunlight"], alpha=0.05, s=2, color="red")
axes[0, 1].set_xlabel("Temperature (°C)")
axes[0, 1].set_ylabel("Sunlight (W/m²)")
axes[0, 1].set_title("Hotter = Generally more sunlight")

# Humidity vs Sunlight
axes[1, 0].scatter(daytime["humidity"], daytime["sunlight"], alpha=0.05, s=2, color="blue")
axes[1, 0].set_xlabel("Humidity (%)")
axes[1, 0].set_ylabel("Sunlight (W/m²)")
axes[1, 0].set_title("More humid = Less sunlight")

# Hour vs Sunlight
hourly_avg = daytime.groupby("hour")["sunlight"].mean()
axes[1, 1].bar(hourly_avg.index, hourly_avg.values, color="orange")
axes[1, 1].set_xlabel("Hour of Day")
axes[1, 1].set_ylabel("Average Sunlight (W/m²)")
axes[1, 1].set_title("Sunlight peaks around noon")

plt.tight_layout()
plt.savefig("../data/processed/feature_relationships.png", dpi=150)
plt.show()

print("These relationships are what the ML model will learn automatically.")
print("You can already SEE the patterns — the model just does it mathematically.")

---
## 1.10 Save clean data

Save the final cleaned dataset. This is what we'll use for model training.

In [ ]:
# Save to CSV
df.to_csv("../data/processed/clean_data.csv")

print("Clean data saved!")
print(f"\nFinal dataset:")
print(f"  Rows:    {len(df)} (one per hour)")
print(f"  Columns: {list(df.columns)}")
print(f"  Period:  {df.index[0].date()} to {df.index[-1].date()}")
print(f"\nTarget:   'sunlight' (what we predict)")
print(f"Features: 'clouds', 'temperature', 'humidity', 'rain', 'hour'")
print(f"\nNext step: Open notebook 02 to train the model!")